In [3]:
from pathlib import Path
import os
import requests
import pandas as pd
import xml.etree.ElementTree as ET

from dotenv import load_dotenv


# ============================================================
# 경로 / API 설정
# ============================================================

PROJECT_DIR = Path.cwd().parents[1]

DATA_ROOT = PROJECT_DIR / "data"
DATA_DIR = DATA_ROOT / "v2_5year_seoul"

RAW_DIR = DATA_DIR / "raw" / "rent"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

load_dotenv(
    PROJECT_DIR / ".env"
)

SERVICE_KEY = os.getenv(
    "MOLIT_RENT_API_KEY"
)

BASE_URL = (
    "https://apis.data.go.kr/1613000/"
    "RTMSDataSvcAptRent/"
    "getRTMSDataSvcAptRent"
)

if not SERVICE_KEY:
    raise ValueError(
        "MOLIT_RENT_API_KEY가 없습니다."
    )

In [2]:
# ============================================================
# API 단건 테스트
# ============================================================

params = {
    "serviceKey": SERVICE_KEY,
    "LAWD_CD": "11680",     # 강남구
    "DEAL_YMD": "202606",
    "pageNo": 1,
    "numOfRows": 100,
}

response = requests.get(
    BASE_URL,
    params=params,
    timeout=30,
)

print("status:", response.status_code)
print(response.text[:1000])

status: 200
<?xml version="1.0" encoding="utf-8" standalone="yes"?><response><header><resultCode>000</resultCode><resultMsg>OK</resultMsg></header><body><items><item><aptNm>아카데미스위트1</aptNm><aptSeq>11680-566</aptSeq><buildYear>2004</buildYear><contractTerm> </contractTerm><contractType> </contractType><dealDay>25</dealDay><dealMonth>6</dealMonth><dealYear>2026</dealYear><deposit>70,000</deposit><excluUseAr>122.07</excluUseAr><floor>7</floor><jibun>467-7</jibun><monthlyRent>230</monthlyRent><preDeposit> </preDeposit><preMonthlyRent> </preMonthlyRent><roadnm>언주로30길 21</roadnm><roadnmbcd></roadnmbcd><roadnmbonbun>00021</roadnmbonbun><roadnmbubun>00000</roadnmbubun><roadnmcd>4166604</roadnmcd><roadnmseq>0</roadnmseq><roadnmsggcd>11680</roadnmsggcd><sggCd>11680</sggCd><umdNm>도곡동</umdNm><useRRRight> </useRRRight></item><item><aptNm>개나리푸르지오</aptNm><aptSeq>11680-3678</aptSeq><buildYear>2006</buildYear><contractTerm> </contractTerm><contractType> </contractType><dealDay>13</dealDay><dealMonth>6<

In [3]:
import xml.etree.ElementTree as ET


def parse_rent_xml(content):
    root = ET.fromstring(content)

    result_code = root.findtext(".//resultCode")
    result_msg = root.findtext(".//resultMsg")

    if result_code != "000":
        raise ValueError(
            f"API 오류: {result_code} / {result_msg}"
        )

    rows = []

    for item in root.findall(".//item"):
        row = {
            child.tag: (
                child.text.strip()
                if child.text
                else None
            )
            for child in item
        }

        rows.append(row)

    return pd.DataFrame(rows)


test_df = parse_rent_xml(
    response.content
)

print("조회 건수:", len(test_df))
print("컬럼 수:", len(test_df.columns))

display(test_df.head())

print(test_df.columns.tolist())

조회 건수: 100
컬럼 수: 25


,aptNm,aptSeq,buildYear,contractTerm,contractType,dealDay,dealMonth,dealYear,deposit,excluUseAr,...,roadnm,roadnmbcd,roadnmbonbun,roadnmbubun,roadnmcd,roadnmseq,roadnmsggcd,sggCd,umdNm,useRRRight
0,아카데미스위트1,11680-566,2004,,,25,6,2026,"70,000",122.07,...,언주로30길 21,None,00021,00000,4166604,0,11680,11680,도곡동,
1,개나리푸르지오,11680-3678,2006,,,13,6,2026,"140,000",148.8017,...,역삼로 314,0,00314,00000,3122008,1,11680,11680,역삼동,
2,강남자곡아이파크,11680-4332,2014,,,27,6,2026,"82,000",74.97,...,자곡로 175,0,00175,00000,3122014,2,11680,11680,자곡동,
3,도곡렉슬,11680-3623,2006,,,6,6,2026,"155,000",84.9984,...,선릉로 221,0,00221,00000,3122006,8,11680,11680,도곡동,
4,현대13차(208~211동),11680-427,1983,,,27,6,2026,"80,000",105.31,...,압구정로29길 23,0,00023,00000,4166514,1,11680,11680,압구정동,


['aptNm', 'aptSeq', 'buildYear', 'contractTerm', 'contractType', 'dealDay', 'dealMonth', 'dealYear', 'deposit', 'excluUseAr', 'floor', 'jibun', 'monthlyRent', 'preDeposit', 'preMonthlyRent', 'roadnm', 'roadnmbcd', 'roadnmbonbun', 'roadnmbubun', 'roadnmcd', 'roadnmseq', 'roadnmsggcd', 'sggCd', 'umdNm', 'useRRRight']


In [1]:
def parse_rent_xml(content):
    root = ET.fromstring(content)

    result_code = root.findtext(".//resultCode")
    result_msg = root.findtext(".//resultMsg")

    if result_code != "000":
        raise ValueError(f"API 오류: {result_code} / {result_msg}")

    rows = []

    for item in root.findall(".//item"):
        rows.append({
            child.tag: child.text.strip() if child.text else None
            for child in item
        })

    total_count = int(root.findtext(".//totalCount") or 0)

    return rows, total_count

In [4]:
SEOUL_GU_CODES = {
    "종로구": "11110",
    "중구": "11140",
    "용산구": "11170",
    "성동구": "11200",
    "광진구": "11215",
    "동대문구": "11230",
    "중랑구": "11260",
    "성북구": "11290",
    "강북구": "11305",
    "도봉구": "11320",
    "노원구": "11350",
    "은평구": "11380",
    "서대문구": "11410",
    "마포구": "11440",
    "양천구": "11470",
    "강서구": "11500",
    "구로구": "11530",
    "금천구": "11545",
    "영등포구": "11560",
    "동작구": "11590",
    "관악구": "11620",
    "서초구": "11650",
    "강남구": "11680",
    "송파구": "11710",
    "강동구": "11740",
}

MONTHS = pd.period_range(
    "2021-01",
    "2026-06",
    freq="M",
).strftime("%Y%m")

print("자치구:", len(SEOUL_GU_CODES))
print("개월:", len(MONTHS))
print("총 수집 대상:", len(SEOUL_GU_CODES) * len(MONTHS))

자치구: 25
개월: 66
총 수집 대상: 1650


In [5]:
def fetch_rent_month(lawd_cd, deal_ymd, num_rows=1000):
    page = 1
    all_rows = []

    while True:
        params = {
            "serviceKey": SERVICE_KEY,
            "LAWD_CD": lawd_cd,
            "DEAL_YMD": deal_ymd,
            "pageNo": page,
            "numOfRows": num_rows,
        }

        response = requests.get(
            BASE_URL,
            params=params,
            timeout=30,
        )
        response.raise_for_status()

        rows, total_count = parse_rent_xml(response.content)

        if not rows:
            break

        all_rows.extend(rows)

        if len(all_rows) >= total_count:
            break

        page += 1

    return pd.DataFrame(all_rows)

In [8]:
import time

failed = []

for gu_name, lawd_cd in SEOUL_GU_CODES.items():
    for deal_ymd in MONTHS:
        file_path = RAW_DIR / f"rent_{lawd_cd}_{deal_ymd}.csv"

        # 이미 받은 파일은 다시 요청하지 않음
        if file_path.exists():
            continue

        try:
            month_df = fetch_rent_month(
                lawd_cd=lawd_cd,
                deal_ymd=deal_ymd,
            )

            month_df["수집_자치구"] = gu_name
            month_df["수집_시군구코드"] = lawd_cd
            month_df["수집_거래연월"] = deal_ymd

            month_df.to_csv(
                file_path,
                index=False,
                encoding="utf-8-sig",
            )

            print(
                f"[{gu_name} {deal_ymd}] "
                f"{len(month_df):,}건"
            )

            time.sleep(0.1)

        except Exception as e:
            failed.append({
                "자치구": gu_name,
                "시군구코드": lawd_cd,
                "거래연월": deal_ymd,
                "오류": str(e),
            })

            print(f"[실패] {gu_name} {deal_ymd}: {e}")

[용산구 202205] 909건
[용산구 202206] 597건
[용산구 202212] 1,010건


In [9]:
expected = len(SEOUL_GU_CODES) * len(MONTHS)

files = list(RAW_DIR.glob("rent_*.csv"))

print("예상 파일:", expected)
print("실제 파일:", len(files))
print("실패 건수:", len(failed))

예상 파일: 1650
실제 파일: 1650
실패 건수: 0


In [10]:
rent_files = sorted(RAW_DIR.glob("rent_*.csv"))

rent_raw = pd.concat(
    [pd.read_csv(file, dtype=str) for file in rent_files],
    ignore_index=True,
)

RENT_RAW_PATH = PROCESSED_DIR / "apt_rent_raw.parquet"

rent_raw.to_parquet(RENT_RAW_PATH, index=False)

print("파일 수:", len(rent_files))
print("전체 shape:", rent_raw.shape)
print("저장 경로:", RENT_RAW_PATH)

파일 수: 1650
전체 shape: (1415810, 28)
저장 경로: c:\Users\uk\Apartment price\data\v2_5year_seoul\processed\apt_rent_raw.parquet
